In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import Dataset, DataLoader
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
data = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train_df, test_df = train_test_split(data, train_size=0.8, random_state=42)
train_df = train_df.to_numpy()
test_df = test_df.to_numpy()
y_train, x_train = train_df[:, 0], train_df[:, 1:].astype(np.float32)
y_test, x_test = test_df[:, 0], test_df[:, 1:].astype(np.float32)

In [ ]:
x_train, y_train, x_test, y_test = map(torch.tensor, (x_train, y_train, x_test, y_test))

In [ ]:
x_train.size()

In [ ]:
eps = 1e-6
def relu(x):
    return x.clamp(0)


def batch_norm(x, gamma):
    batch_mean = x.mean(dim=0)
    batch_var = (x - batch_mean).var(dim=0)
    x_norm = (x - batch_mean) / torch.sqrt(batch_var + eps)
    return gamma * x_norm

In [ ]:
BATCH_SIZE = 64
epochs = 300
EPOCHS_PER_EVAL = 10
lr = 1e-2
hidden_features = 512

In [ ]:
x_train, y_train, x_test, y_test = [t.to(device) for t in [x_train, y_train, x_test, y_test]]
w1 = torch.distributions.Uniform(-np.sqrt(1/784), np.sqrt(1/784)).sample((hidden_features, 784)).to(device).requires_grad_()
b1 = torch.distributions.Uniform(-np.sqrt(1/784), np.sqrt(1/784)).sample((hidden_features, )).to(device).requires_grad_()
w2 = torch.distributions.Uniform(-np.sqrt(1/hidden_features), np.sqrt(1/hidden_features)).sample((hidden_features, hidden_features)).to(device).requires_grad_()
b2 = torch.distributions.Uniform(-np.sqrt(1/hidden_features), np.sqrt(1/hidden_features)).sample((hidden_features, )).to(device).requires_grad_()
w3 = torch.distributions.Uniform(-np.sqrt(1/hidden_features), np.sqrt(1/hidden_features)).sample((10, hidden_features)).to(device).requires_grad_()
b3 = torch.distributions.Uniform(-np.sqrt(1/hidden_features), np.sqrt(1/hidden_features)).sample((10, )).to(device).requires_grad_()


In [ ]:
class MNIST(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self. n = len(x)
    def __len__(self) :
        return self.n
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

train_ds = MNIST(x_train, y_train)
test_ds = MNIST(x_test, y_test)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE)

In [ ]:
def softmax(x):
    x_exp = torch.exp(x - x.max(dim=1, keepdim=True).values)
    return x_exp / x_exp.sum(dim=1, keepdim=True)


def forward(x):
    x = x / 255.0
    h1 = relu(x @ w1.T + b1)
    h1 = batch_norm(h1, gamma=1.0)
    h2 = relu(h1 @ w2.T + b2)
    h2 = batch_norm(h2, gamma=1.0)
    logits = h2 @ w3.T + b3
    return logits


def cross_entropy_loss(logits, y):
    probs = softmax(logits)
    log_likelihood = -torch.log(probs[range(len(y)), y] + eps)
    return log_likelihood.mean()


def accuracy(x, y):
    with torch.no_grad():
        logits = forward(x)
        preds = torch.argmax(logits, dim=1)
        return (preds == y).float().mean().item()


params = [w1, b1, w2, b2, w3, b3]
optimizer = optim.SGD(params, lr=lr)

for epoch in range(1, epochs + 1):
    total_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)

        logits = forward(xb)
        loss = cross_entropy_loss(logits, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_dl)
    acc = accuracy(x_test, y_test)
    print(f"Epoch [{epoch:03d}/{epochs}] | Loss: {avg_loss:.4f} | Test Accuracy: {acc*100:.2f}%")

print("\nFinal Custom MLP Test Accuracy: {:.2f}%".format(accuracy(x_test, y_test) * 100))


In [ ]:
# Built-in MLP

class TorchMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, hidden_features),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_features),
            nn.Linear(hidden_features, hidden_features),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_features),
            nn.Linear(hidden_features, 10)
        )

    def forward(self, x):
        return self.net(x / 255.0)

model = TorchMLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=lr)

for epoch in range(1, epochs + 1):
    total_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if epoch % EPOCHS_PER_EVAL == 0:
        with torch.no_grad():
            preds = torch.argmax(model(x_test), dim=1)
            acc = (preds == y_test).float().mean().item()
        print(f"[Built-in] Epoch {epoch}/{epochs} | Loss: {total_loss/len(train_dl):.4f} | Test Acc: {acc:.4f}")


with torch.no_grad():
    preds = torch.argmax(model(x_test), dim=1)
    built_in_acc = (preds == y_test).float().mean().item()
print(f"\nFinal Built-in MLP Test Accuracy: {built_in_acc:.4f}")


## Final Observation
The Results of the two methods are almost identical. The values are very close for each iteration